# RAG Inference Pipeline

**Strategy:** BM25 (sparse) + Dense (FAISS) → RRF fusion → Cross-encoder rerank

| Stage | Method | Output |
|-------|--------|--------|
| 1a | BM25Okapi | top-`bm25_top_k` candidates |
| 1b | Dense (Qwen3-Embedding + FAISS FlatIP) | top-`dense_top_k` candidates |
| 2 | RRF fusion | merged ranking |
| 3 | Cross-encoder rerank | final top-`final_top_k` results |

All tunable parameters are in the **Config** cell below.

In [1]:
import sys, json
from pathlib import Path

# ── project root on path ──────────────────────────────────────────────────────
ROOT = Path("../").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

Project root: /home/arkkuma/final


## Config — adjust all key parameters here

In [20]:
# ── artifact paths ────────────────────────────────────────────────────────────
FAISS_MANIFEST = ROOT / "artifacts/faiss/chunks_embedding.Qwen__Qwen3-Embedding-0.6B.nlp_prefix.FlatIP.manifest.json"
META_JSONL     = ROOT / "artifacts/faiss/chunks_embedding.Qwen__Qwen3-Embedding-0.6B.nlp_prefix.FlatIP.meta.jsonl"
BENCHMARK      = ROOT / "benchmark/benchmark.json"

# ── models ───────────────────────────────────────────────────────────────────
# HuggingFace ID or local path; pipeline resolves model_checkpoints/ automatically
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# ── retrieval hyper-parameters ────────────────────────────────────────────────
BM25_TOP_K      = 100   # BM25 candidates per query
DENSE_TOP_K     = 100   # Dense candidates per query
RRF_K           = 60    # RRF smoothing constant
RRF_WEIGHTS     = [1.0, 1.0]   # [bm25_weight, dense_weight]
RERANK_INPUT_K  = 30    # candidates fed to cross-encoder
FINAL_TOP_K     = 5     # results returned per query

# ── evaluation ────────────────────────────────────────────────────────────────
EVAL_KS         = [1, 3, 5, 10]   # cut-offs for Recall@k and NDCG@k
EVAL_VERBOSE    = False            # print per-question hit/miss

## 1. Load Pipeline

In [5]:
from retrieval.pipeline import RetrievalPipeline

pipeline = RetrievalPipeline(
    meta_jsonl_path     = META_JSONL,
    faiss_manifest_path = FAISS_MANIFEST,
    cross_encoder_model = CROSS_ENCODER_MODEL,
    bm25_top_k          = BM25_TOP_K,
    dense_top_k         = DENSE_TOP_K,
    rrf_k               = RRF_K,
    rrf_weights         = RRF_WEIGHTS,
    rerank_input_k      = RERANK_INPUT_K,
    final_top_k         = FINAL_TOP_K,
)
print("Pipeline ready.")

[BM25] Indexed 1404 docs from chunks_embedding.Qwen__Qwen3-Embedding-0.6B.nlp_prefix.FlatIP.meta.jsonl
[Dense] FAISS loaded: 1404 vectors, dim=1024 from chunks_embedding.Qwen__Qwen3-Embedding-0.6B.nlp_prefix.FlatIP.index
[Reranker] Using HuggingFace: cross-encoder/ms-marco-MiniLM-L-6-v2
[Reranker] Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline ready.


## 3. Single-Query Debug View

Inspect scores and metadata at each stage.

In [6]:
DEBUG_QUERY = "What is chicken teriyaki?"

# Stage 1a: BM25
bm25_hits = pipeline.bm25.search(DEBUG_QUERY, top_k=10)
print("── BM25 top-5 ──")
for vid, score, meta in bm25_hits[:5]:
    print(f"  [{vid:4d}] score={score:.4f}  {meta['title']} | {meta.get('sections',[])}")

print()

# Stage 1b: Dense
dense_hits = pipeline.dense.search(DEBUG_QUERY, top_k=10)
print("── Dense top-5 ──")
for vid, score, meta in dense_hits[:5]:
    print(f"  [{vid:4d}] score={score:.4f}  {meta['title']} | {meta.get('sections',[])}")

print()

# Stage 2: RRF
from retrieval.fusion import rrf_fuse
fused = rrf_fuse(
    [bm25_hits, dense_hits],
    k=RRF_K,
    weights=RRF_WEIGHTS,
)
print("── RRF fused top-5 ──")
for vid, score, meta in fused[:5]:
    print(f"  [{vid:4d}] rrf={score:.6f}  {meta['title']} | {meta.get('sections',[])}")

print()

# Stage 3: Rerank
reranked = pipeline.reranker.rerank(DEBUG_QUERY, fused[:RERANK_INPUT_K], top_k=FINAL_TOP_K)
print(f"── Cross-encoder top-{FINAL_TOP_K} ──")
for vid, score, meta in reranked:
    print(f"  [{vid:4d}] ce={score:.4f}  {meta['title']} | {meta.get('sections',[])}")
    print(f"          {meta['text'][:120]}...")

── BM25 top-5 ──
  [ 951] score=12.0513  Japanese cuisine | ['Outside Japan (part 6)']
  [ 241] score=10.4655  Cookbook:Teriyaki Sauce | ['Lead', 'Ingredients', 'Procedure']
  [ 169] score=10.1279  Cookbook:Steamed Buns with BBQ Pork | ['Lead', 'Ingredients', 'Bun filling', 'Steamed buns']
  [ 735] score=9.9705  Taiwanese cuisine | ['Food of the Taiwanese Aborigines (part 1)']
  [ 965] score=9.3232  Japanese cuisine | ['Indonesia (part 3)']

[Dense] Local model not found, using HuggingFace: Qwen/Qwen3-Embedding-0.6B
[Dense] Loading query encoder: Qwen/Qwen3-Embedding-0.6B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

── Dense top-5 ──
  [ 241] score=0.6629  Cookbook:Teriyaki Sauce | ['Lead', 'Ingredients', 'Procedure']
  [   3] score=0.5918  12. Japan | ['Bento box (part 3)']
  [1053] score=0.5368  Nagoya cuisine | ['Lead (part 1)']
  [ 942] score=0.5345  Japanese cuisine | ['Chūka ryōri—Japanese Chinese cuisine (part 2)']
  [ 963] score=0.5317  Japanese cuisine | ['Indonesia (part 1)']

── RRF fused top-5 ──
  [ 241] rrf=0.032522  Cookbook:Teriyaki Sauce | ['Lead', 'Ingredients', 'Procedure']
  [ 951] rrf=0.016393  Japanese cuisine | ['Outside Japan (part 6)']
  [   3] rrf=0.016129  12. Japan | ['Bento box (part 3)']
  [ 169] rrf=0.015873  Cookbook:Steamed Buns with BBQ Pork | ['Lead', 'Ingredients', 'Bun filling', 'Steamed buns']
  [1053] rrf=0.015873  Nagoya cuisine | ['Lead (part 1)']

── Cross-encoder top-5 ──
  [ 241] ce=5.4177  Cookbook:Teriyaki Sauce | ['Lead', 'Ingredients', 'Procedure']
          | Sauces | Meat Recipes

The word teriyaki is a combination of two Japanese words, teri , "lu

## 4. Evaluate on Benchmark

Ground-truth matching: title + section (strips `(part N)` suffixes).

In [21]:
from evaluation.metrics import load_benchmark, evaluate, print_metrics

benchmark = load_benchmark(BENCHMARK)
print(f"Benchmark: {len(benchmark)} questions")

Benchmark: 50 questions


In [22]:
metrics = evaluate(
    pipeline,
    benchmark,
    ks=EVAL_KS,
    verbose=EVAL_VERBOSE,
)

print_metrics(metrics)


  Metric              Score
----------------------------------------
  MRR                0.8057
  Recall@1           0.7000
  NDCG@1             0.7000
  Recall@3           0.8800
  NDCG@3             0.8109
  Recall@5           0.9600
  NDCG@5             0.8445
  Recall@10          0.9600
  NDCG@10            0.8445


## 5. Ablation: BM25-only vs Dense-only vs BM25+Dense+Rerank

In [23]:
from evaluation.metrics import recall_at_k, reciprocal_rank, ndcg_at_k

ABLATION_K = 5   # cut-off for ablation comparison

def _eval_single_stage(retrieve_fn, benchmark, k=ABLATION_K):
    """Evaluate a single-stage retriever (BM25-only or Dense-only)."""
    rr_list, rec_list, ndcg_list = [], [], []
    for item in benchmark:
        # Single-stage retrievers return raw meta dicts that contain vector_id
        metas = retrieve_fn(item["query"])
        gt_id = item["gt_chunk_id"]
        rr_list.append(reciprocal_rank(metas, gt_id))
        rec_list.append(recall_at_k(metas, gt_id, k))
        ndcg_list.append(ndcg_at_k(metas, gt_id, k))
    n = len(benchmark)
    return {
        "MRR":          sum(rr_list)  / n,
        f"Recall@{k}":  sum(rec_list) / n,
        f"NDCG@{k}":    sum(ndcg_list)/ n,
    }

def _bm25_retrieve(query):
    hits = pipeline.bm25.search(query, top_k=max(EVAL_KS))
    return [meta for _, _, meta in hits]

def _dense_retrieve(query):
    hits = pipeline.dense.search(query, top_k=max(EVAL_KS))
    return [meta for _, _, meta in hits]

print("Evaluating BM25-only ...")
m_bm25  = _eval_single_stage(_bm25_retrieve, benchmark)

print("Evaluating Dense-only ...")
m_dense = _eval_single_stage(_dense_retrieve, benchmark)

print(f"\n── Ablation @ k={ABLATION_K} ────────────────────────")
print(f"{'Metric':<15} {'BM25':>8} {'Dense':>8} {'Full Pipeline':>14}")
print("-" * 50)
for key in m_bm25:
    print(f"{key:<15} {m_bm25[key]:>8.4f} {m_dense[key]:>8.4f} {metrics.get(key, 0):>14.4f}")

Evaluating BM25-only ...
Evaluating Dense-only ...

── Ablation @ k=5 ────────────────────────
Metric              BM25    Dense  Full Pipeline
--------------------------------------------------
MRR               0.6490   0.8062         0.8057
Recall@5          0.7800   0.9400         0.9600
NDCG@5            0.6769   0.8359         0.8445


## 6. Generation — Qwen2.5-0.5B-Instruct

Load the generator, edit `SYSTEM_PROMPT` to change the model's behaviour.

In [ ]:
# ── Generation Config ─────────────────────────────────────────────────────────

# HuggingFace ID or local path (model_checkpoints/ is checked first)
GENERATOR_MODEL    = "Qwen/Qwen2.5-0.5B-Instruct"

MAX_NEW_TOKENS     = 256    # token budget per response
CONTEXT_MAX_CHARS  = 2000   # character cap for the context block in the prompt

# ── System Prompt — edit freely ───────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are a knowledgeable culinary assistant. \
Answer the user's question using only the information provided in the context. \
Be concise and factual. \
If the context does not contain enough information to answer, say so clearly \
and do not make up details.\
"""

In [25]:
from generation.generator import RAGGenerator

generator = RAGGenerator(
    model_ref         = GENERATOR_MODEL,
    max_new_tokens    = MAX_NEW_TOKENS,
    context_max_chars = CONTEXT_MAX_CHARS,
)

[Generator] Using HuggingFace: Qwen/Qwen2.5-0.5B-Instruct
[Generator] Loading model on cuda: Qwen/Qwen2.5-0.5B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[Generator] Ready.


In [26]:
# ── Single-Query Generation Test ──────────────────────────────────────────────
GEN_QUERY = "What is chicken teriyaki?"

context_chunks = pipeline.retrieve(GEN_QUERY)

print(f"Query : {GEN_QUERY}")
print(f"Context chunks used: {len(context_chunks)}")
for c in context_chunks:
    print(f"  [{c['doc_id']}] {c['title']} | {c['sections']} | score={c['score']:.4f}")

print("\n── Generated Response ──")
response = generator.generate(
    query          = GEN_QUERY,
    context_chunks = context_chunks,
    system_prompt  = SYSTEM_PROMPT,
)
print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Query : What is chicken teriyaki?
Context chunks used: 5
  [000] Cookbook:Teriyaki Sauce | ['Lead', 'Ingredients', 'Procedure'] | score=5.4177
  [001] 12. Japan | ['Bento box (part 3)'] | score=4.5197
  [002] Japanese cuisine | ['Indonesia (part 1)'] | score=3.6653
  [003] Japanese cuisine | ['Outside Japan (part 4)'] | score=2.4556
  [004] Japanese cuisine | ['Indonesia (part 3)'] | score=1.4063

── Generated Response ──
Chicken teriyaki is a technique involving grilling food in a marinade of soy sauce, mirin, sake, onion, ginger, and other ingredients such as salad, rice, prawn gyoza, edamame beans, and pickled ginger. This method allows for the preparation of raw meat or vegetables before marinating them in the marinade.


In [29]:
# ── Full End-to-End: Retrieve → Generate ──────────────────────────────────────
# Runs the complete pipeline on input_payload_sample.json

INPUT_FILE  = ROOT / "benchmark.json"
OUTPUT_FILE = ROOT / "output.json"   # ← saved here

with open(INPUT_FILE, encoding="utf-8") as f:
    input_payload = json.load(f)

# Step 1: retrieve context for every query
retrieval_output = pipeline.run_batch(input_payload)

# Step 2: fill in responses
final_output = generator.fill_responses(
    pipeline_output = retrieval_output,
    system_prompt   = SYSTEM_PROMPT,
)

# Step 3: save to project root
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)
print(f"Saved → {OUTPUT_FILE}")

print("\n── Final Output Payload ──")
print(json.dumps(final_output, indent=2, ensure_ascii=False))

  Generating [1/50] What cooking technique is used to make tempura?...
  Generating [2/50] What is chicken teriyaki?...
  Generating [3/50] What is dashi traditionally made with?...
  Generating [4/50] Where did takoyaki originate?...
  Generating [5/50] What is inside a takoyaki?...
  Generating [6/50] What is xiǎochī in Taiwanese cuisine?...
  Generating [7/50] Where is the Taiwanese oyster omelette thought to have origi...
  Generating [8/50] What gives the Taiwanese oyster omelette its thick consisten...
  Generating [9/50] What principle do northern Chinese apply to food preparation...
  Generating [10/50] Why is hot pot considered a healthy meal?...
  Generating [11/50] What does 'yum cha' mean in Cantonese?...
  Generating [12/50] What is a buuz typically filled with?...
  Generating [13/50] Why is rice not grown locally in Mongolia?...
  Generating [14/50] What type of rice is required to make onigiri?...
  Generating [15/50] What happens if spicy miso udon soup reaches a boil?

## 7. Generation Quality Evaluation

Evaluates responses in `output_payload.json` against benchmark ground-truth answers.

| Metric | Description |
|--------|-------------|
| **Token F1** | Unigram overlap (SQuAD-style): precision / recall / F1 after lowercasing + punctuation removal |
| **BERTScore** | Contextual embedding similarity via `roberta-large` (or configurable model) |

In [ ]:
from evaluation.gen_metrics import evaluate_generation, print_gen_metrics
from evaluation.metrics import load_benchmark

# ── Generation Evaluation Config ──────────────────────────────────────────────
# BERTScore backbone: "roberta-large" (best quality) or "distilbert-base-uncased" (faster on CPU)
BERTSCORE_MODEL    = "distilbert-base-uncased"
GEN_EVAL_VERBOSE   = False
# Load output
OUTPUT_FILE = ROOT / "output.json"
with open(OUTPUT_FILE, encoding="utf-8") as f:
    output_payload = json.load(f)

# Load benchmark (supports both old and new schema automatically)
eval_benchmark = load_benchmark(BENCHMARK)
print(f"Output items  : {len(output_payload['results'])}")
print(f"Benchmark items: {len(eval_benchmark)}")

Output items  : 50
Benchmark items: 50


In [ ]:


# fallback defaults if Config cell was not re-run
_bertscore_model = globals().get("BERTSCORE_MODEL", "distilbert-base-uncased")
_gen_verbose     = globals().get("GEN_EVAL_VERBOSE", False)

gen_results = evaluate_generation(
    output_payload  = output_payload,
    benchmark       = eval_benchmark,
    bertscore_model = _bertscore_model,
    verbose         = _gen_verbose,
)

print_gen_metrics(gen_results)

[BERTScore] Scoring 50 pairs with distilbert-base-uncased ...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  Generation Evaluation  (n=50)
  Metric                    Precision    Recall        F1
  ------------------------------------------------
  Token F1                     0.4981    0.6115    0.5043
  BERTScore                    0.8784    0.9009    0.8881


In [37]:
# ── Per-query breakdown (worst → best by Token F1) ────────────────────────────
per_item = sorted(gen_results["per_item"], key=lambda x: x["token_f1"]["f1"])

print(f"{'#':>3}  {'Token F1':>9}  {'BERT-F1':>9}  Query / Prediction vs Reference")
print("-" * 100)
for item in per_item:
    tf1 = item["token_f1"]["f1"]
    bf1 = item["bert_score"]["f1"]
    q   = item["query"][:55]
    print(f"     {tf1:>9.4f}  {bf1:>9.4f}  Q: {q}")
    print(f"     {'':9}  {'':9}  P: {item['prediction'][:90]}")
    print(f"     {'':9}  {'':9}  R: {item['reference'][:90]}")
    print()

  #   Token F1    BERT-F1  Query / Prediction vs Reference
----------------------------------------------------------------------------------------------------
        0.1000     0.8102  Q: What are Xiaolong Bao?
                           P: Xiaolongbao is a type of steamed bun from the Jiangnan region that is popular in Shanghai.
                           R: Xiaolong Bao are 'little dragon' dumplings, one of the most famous baozi fillings, usually

        0.1250     0.7813  Q: Why is hot pot considered a healthy meal?
                           P: Hot pot is considered a healthy meal because it combines elements of traditional Chinese c
                           R: Hot pot is considered healthy because it is balanced, warming, and cooked without oil or f

        0.1250     0.8001  Q: What is the purpose of nori in onigiri?
                           P: Nori serves as a protective layer between the rice and filling, helping to prevent the ric
                           R: Onigiri 